# Course 4: Mathematics and AI
## Session 4: Mechanistic interpretability with mathematics: grokking

**Goal:** train a small network on addition modulo a prime, watch it generalize long after it has memorized, and find the Fourier features it learned.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. The task
Learn `(a + b) mod p` from half of all pairs. The other half is the test. Nanda et al. (2023) used a one-layer transformer and p = 113; to keep this to about a minute on a CPU we use p = 47 and a network with a learned embedding and one hidden layer. The phenomenon is the same.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
p = 47
pairs = torch.tensor([(a, b) for a in range(p) for b in range(p)]); y = (pairs[:, 0] + pairs[:, 1]) % p
perm = torch.randperm(p * p); tr, te = perm[:p * p // 2], perm[p * p // 2:]
emb = nn.Embedding(p, 64); net = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, p))
model = lambda idx: net(emb(pairs[idx]).flatten(1))
opt = torch.optim.AdamW([*emb.parameters(), *net.parameters()], lr=1e-3, weight_decay=1.0)     # strong weight decay matters

history, snapshot = [], None
for step in range(8001):
    loss = F.cross_entropy(model(tr), y[tr]); opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        with torch.no_grad():
            a_tr = (model(tr).argmax(1) == y[tr]).float().mean().item(); a_te = (model(te).argmax(1) == y[te]).float().mean().item()
            norm = sum((w ** 2).sum() for w in [*emb.parameters(), *net.parameters()]).sqrt().item()
        history.append((step, a_tr, a_te, norm)); print(f"step {step:5d}   train accuracy {a_tr:.2f}   test accuracy {a_te:.2f}   weight norm {norm:6.1f}")
    if step == 500: snapshot = emb.weight.detach().clone()                                      # the memorizing network

### 2. Read the curve
Training accuracy reaches 100% within a few hundred steps while test accuracy is still zero: the network has memorized. Thousands of steps later the test accuracy climbs. Nothing changed in the data or the loss. Weight decay keeps shrinking the weights, and the memorizing solution is more expensive in weight norm than the general one.

In [ ]:
import matplotlib.pyplot as plt
steps, a_tr, a_te, norms = map(np.array, zip(*history))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(steps, a_tr, label="train"); ax[0].plot(steps, a_te, label="test"); ax[0].set_xlabel("step"); ax[0].set_ylabel("accuracy"); ax[0].legend()
ax[1].plot(steps, norms); ax[1].set_xlabel("step"); ax[1].set_ylabel("weight norm"); fig.savefig("grokking.png")
memorized_at = steps[np.argmax(a_tr > 0.99)]; generalized_at = steps[np.argmax(a_te > 0.9)] if (a_te > 0.9).any() else None
print("memorized by step", memorized_at, "| generalized (test above 90%) by step", generalized_at)
assert a_te[1] < 0.1 and a_te[-1] > 0.9 and generalized_at > 4 * max(memorized_at, 500)

### 3. What did it learn? Look at the embedding in the Fourier basis
Addition mod p is addition of angles on a circle. If the network represents the number a as `cos(2 pi k a / p)` and `sin(2 pi k a / p)` for a few frequencies k, it can add with trigonometric identities. Take the Fourier transform of the embedding along the number axis and see which frequencies carry the energy.

In [ ]:
def spectrum(E):
    Fh = torch.fft.rfft(E - E.mean(0, keepdim=True), dim=0)            # along the number axis
    power = (Fh.abs() ** 2).sum(1)[1:]                                  # drop frequency 0
    return (power / power.sum()).numpy()

before, after = spectrum(snapshot), spectrum(emb.weight.detach())
top = np.argsort(-after)[:6] + 1
print("share of energy in the 6 strongest of 23 frequencies:  memorizing network %.2f | after grokking %.2f" % (np.sort(before)[-6:].sum(), np.sort(after)[-6:].sum()))
print("key frequencies after grokking:", sorted(top.tolist()))
fig, ax = plt.subplots(1, 2, figsize=(11, 3)); ax[0].bar(range(1, 24), before); ax[0].set_title("step 500 (memorized)"); ax[1].bar(range(1, 24), after); ax[1].set_title("after grokking")
for a in ax: a.set_xlabel("frequency k")
fig.savefig("fourier.png")
assert np.sort(after)[-6:].sum() > np.sort(before)[-6:].sum() + 0.15

The memorizing network spreads its energy over all frequencies. The network that generalizes concentrates it on a handful. Each seed picks different frequencies; the algorithm is the same.

### 4. A causal test, not just a picture
Keep only the key frequencies in the embedding and throw the rest away. If the network really computes with them, accuracy should survive. Then do the opposite.

In [ ]:
@torch.no_grad()
def accuracy_with(keep):
    E = emb.weight.detach(); mean = E.mean(0, keepdim=True); Fh = torch.fft.rfft(E - mean, dim=0)
    mask = torch.zeros(Fh.size(0), 1); mask[list(keep)] = 1
    E2 = torch.fft.irfft(Fh * mask, n=p, dim=0) + mean
    out = net(E2[pairs[te]].flatten(1)); return (out.argmax(1) == y[te]).float().mean().item()
key = set(top.tolist()); rest = set(range(1, 24)) - key
acc_key, acc_rest = accuracy_with(key), accuracy_with(rest)
print(f"test accuracy with only the 6 key frequencies: {acc_key:.2f} | with only the other 17: {acc_rest:.2f}")
assert acc_key > acc_rest + 0.3

### 5. The identity behind it
`cos(w(a + b)) = cos(wa) cos(wb) - sin(wa) sin(wb)`. The hidden layer multiplies the embeddings of a and b; the output layer reads off which c makes `cos(w(a + b - c))` largest. Nanda et al. traced exactly this circuit in a transformer.

In [ ]:
k_ = int(top[0]); w = 2 * np.pi * k_ / p; a, b = 13, 40
lhs = np.cos(w * (a + b)); rhs = np.cos(w * a) * np.cos(w * b) - np.sin(w * a) * np.sin(w * b)
score = [sum(np.cos(2 * np.pi * k * (a + b - c) / p) for k in top) for c in range(p)]
print("identity holds:", np.isclose(lhs, rhs), "| the c that maximizes the summed cosines:", int(np.argmax(score)), "| (a + b) mod p:", (a + b) % p)
assert int(np.argmax(score)) == (a + b) % p

### TODO
1. Set `weight_decay=0.0` and train again. Does the network ever generalize?
2. Train on 30% and on 70% of the pairs. How does the delay change?
3. Replace the network with a one-layer transformer (attention over the two tokens, then an MLP) and repeat section 3 for p = 113 on a GPU.